# Polygenic score calculation using PRS-CS

In [ ]:
output_dir = os.environ.get('WORKSPACE_BUCKET')


## Download LD reference

In [ ]:
LD_dir = f"{output_dir}/reference/LD_reference"
os.makedirs(LD_dir, exist_ok=True)

os.system(
    f'wget -b -c '
    f'https://personal.broadinstitute.org/hhuang/public/PRS-CSx/Reference/UKBB/ldblk_ukbb_eur.tar.gz '
    f'-P "{LD_dir}"'
)

os.system(
    f'tar -xzf "{LD_dir}/ldblk_ukbb_eur.tar.gz" '
    f'-C "{LD_dir}"'
)

## Download gwas summary

In [ ]:
gwas_dir = f"{output_dir}/GWAS_summary"
os.makedirs(gwas_dir, exist_ok=True)

os.system(
    f'wget -b -c '
    f'xx/traits_gwas.tar.gz '
    f'-P "{gwas_dir}"'
)

os.system(
    f'tar -xzf "{gwas_dir}/traits_gwas.tar.gz" '
    f'-C "{gwas_dir}"'
)

In [ ]:
import pandas as pd
import subprocess
import tempfile
import os
import re
from mpire import WorkerPool
from io import StringIO
import numpy as np
n_threads = "10" # change n_threads value with your computational resource, n_cpus = n_threads * njobs
os.environ["OMP_NUM_THREADS"] = n_threads
os.environ["MKL_NUM_THREADS"] = n_threads
os.environ["OPENBLAS_NUM_THREADS"] = n_threads
os.environ["NUMEXPR_NUM_THREADS"] = n_threads

In [ ]:
def run_prscs_single_chrom(chrom, path_to_prscs, ref_dir, summary_stats, n_gwas, output_dir):

    output_prs = os.path.join(output_dir, f"prs_chr{chrom}")
    
    bim_path = f"{output_dir}/results/qc_merged/merged_genome.bim" # replace your own data path
    
    command = f"python {path_to_prscs} --ref_dir {ref_dir} --bim_file {bim_path} --sst_file {summary_stats} --n_gwas {n_gwas} --chrom={chrom} --out {output_prs}"
    subprocess.run(command, shell=True, check=True) 
    print(f"Chromosome {chrom}: PRScs completed successfully.")

In [ ]:
import os
import glob

def run_prscs(PRScs_path, gwas_file, n_gwas, ref_dir, root_dir, output_prefix, njobs = 1):

    output_dir = root_dir + f"/{output_prefix}"
    os.makedirs(output_dir, exist_ok=True)
    
    chrom_lst = list(range(1, 23))  # Chromosomes 1 to 22 for autosomes
    
    cur_args = [(chrom, PRScs_path, ref_dir, gwas_file, n_gwas, output_dir) for chrom in chrom_lst]
    
    with WorkerPool(n_jobs=njobs) as pool:
        pool.map(run_prscs_single_chrom, cur_args, progress_bar=False)
    
    print(f"All chromosomes have been processed.")

In [ ]:
stats2n_gwas = {'T1_anx2026_b37_beta_p.csv': 851685,
 'T1_bip2025_b3738_beta_p.csv': 458440,
 'T1_ed2025_b3738_beta_p.csv': 458440,
 'T1_id2022_b3738_beta_p.csv': 64641,
 'T1_mdd2025_b37_beta_p.csv': 5053033,
 'T1_ocd2025_b37_beta_p.csv': 1011601,
 'T1_ptsd2024_b37_beta_p.csv': 1280933,
 'T1_scz2025_beta_p.csv': 458440,
 'T1_sud2023_b37_beta_p.csv': 1025550,
 'T2_extraversion2015_b37_beta_p.csv': 63030,
 'T2_loneliness2025_b3738_beta_p.csv': 413936,
 'T2_mtag2025_beta_p.csv': 848919,
 'T2_neuroticism2025_b3738_beta_p.csv': 279645,
 'T2_rtb2023_b3738_beta_p.csv': 274537,
 'T2_swb2024_b3738_beta_p.csv': 358135,
 'T3_cad2025_multi_b3738_beta_p.csv': 429407,
 'T3_chronic_pain_2024_multi_b3738_beta_p.csv': 576720,
 'T3_insomnia2024_multi_b3738_beta_p.csv': 575155,
 'T3_migraine2025_eur_B3738_beta_p.csv': 458440,
 'T3_t2d2025_b3738_beta_p.csv': 418949,
 'T4_bmi2025_b3738_beta_p.csv': 204747,
 'T4_crp2025_multi_b3738_beta_p.csv': 408491,
 'T5_adhd2025_b3738_beta_p.csv': 299017,
 'T5_asd2019_b37_beta_p.csv': 11178,
 'T5_intelligence2025_multi_beta_p.csv': 139298,
 'T5_neuro2025_eur_beta_p.csv': 70560,
 'T5_ts2019_b37_beta_p.csv': 14307,
 'T6_SSRI_swithcing_beta_p.csv': 21228,
 'T6_antidepressant_response_beta_p.csv': 5151,
 'T6_lithium_response_beta_p.csv': 2563
}

In [ ]:
ld_ref = f"{output_dir}/reference/LD_reference/ldblk_ukbb_eur" # replace this path by your LD reference path

PRScs_path = "./PRScs.py"

stats_lst = glob.glob(f"{output_dir}/GWAS_summary/*.csv") # replace this by your own gwas summary path, 
                                                                # the gwas summary are saved in csv file under this path 
filenames = [os.path.basename(f) for f in stats_lst]


In [ ]:
PGS_dir = f"{output_dir}/PRScs" # replace this path by your results path
os.makedirs(PGS_dir, exist_ok=True)

for gwas_file in stats_lst:
    stats_name = gwas_file.split("/")[-1]
    n_gwas = stats2n_gwas[stats_name]
    #run_prscs(PRScs_path, gwas_file, n_gwas, ld_ref, PGS_dir, stats_name, njobs = 4) # change n_threads value with your computational resource, n_cpus = n_threads * njobs

In [ ]:
import os
import glob

def merge_prscs_by_trait_folder(trait_folder, output_filename, chr_range=range(1, 23)):

    output_path = os.path.join(trait_folder, output_filename)
    
    with open(output_path, 'w') as outfile:
        for i, chr_num in enumerate(chr_range, start=1):
            filename = f"prs_chr{chr_num}_pst_eff_a1_b0.5_phiauto_chr{chr_num}.txt"
            file_path = os.path.join(trait_folder, filename)
            
            if not os.path.exists(file_path):
                print(f"[WARNING] File not found: {file_path}")
                continue
            
            with open(file_path, 'r') as infile:
                if i == 1:
                    # Write header for the first file
                    outfile.write(infile.read())
                else:
                    # Skip header for subsequent files
                    next(infile)  # skip header line
                    outfile.write(infile.read())

    print(f"[INFO] Merged file saved to: {output_path}")
    return output_path

In [ ]:
gwas_dir = f"{output_dir}/GWAS_summary"

stats_lst = glob.glob(f"{gwas_dir}/*.csv")

filenames = [os.path.basename(f) for f in stats_lst]

traits_lst = stats_lst#[0:1]#[2:3] #+ stats_lst[3:5] + stats_lst[5:]
traits_lst

In [ ]:
for gwas_file in traits_lst:

    suffix = gwas_file.split("/")[-1]
    
    trait_folder = f"{output_dir}/results/PRScs/{suffix}"

    trait_name = gwas_file.split("/")[-1].split("_beta_p.csv")[0]
    
    output_filename = f"{trait_name}_pst_eff_a1_b0.5_phiauto.txt"
    
    snp_file = merge_prscs_by_trait_folder(trait_folder, output_filename, chr_range=range(1, 23))

    bfile = f"{output_dir}/results/qc_merged/merged_genome"
    
    plink_prs = os.path.join(trait_folder, f"{trait_name}_PGS_PRScs")
    
    command_plink = f"/home/jupyter/workspace/tools/plink2 --bfile {bfile} --score {snp_file} 2 4 6 --out {plink_prs}"
    
    subprocess.run(command_plink, shell=True, check=True)
    
    print(f"PRS Output: {plink_prs}")

# merge phenotype and PGS into one file

In [ ]:
import pandas as pd
pheno_df = pd.read_csv(f"{output_dir}/results/phenotype_all.csv")

pheno_df.drop(columns=['Unnamed: 0'], inplace=True)
cols = ['concurrent_polypharmacy_binary', 'lifetime_polypharmacy_binary']
pheno_df[cols] = pheno_df[cols].astype(int)
pheno_df

In [ ]:
for gwas_file in traits_lst:

    suffix = gwas_file.split("/")[-1]
    
    trait_folder = f"{output_dir}/results/PRScs/{suffix}"

    trait_name = gwas_file.split("/")[-1].split("_beta_p.csv")[0]
    
    prs_file = os.path.join(trait_folder, f"{trait_name}_PGS_PRScs.sscore")
    
    PRS_df = pd.read_csv(prs_file, sep="\t")

    prs_name = f"{trait_name}_PGS_PRScs"
    
    PRS_df = PRS_df.rename(columns={"SCORE1_AVG": prs_name, "#FID": "FID"})
    #print(PRS_df)
    pheno_df = pheno_df.merge(PRS_df[["FID", "IID", prs_name]], on=["FID", "IID"], how="left")

In [ ]:
pheno_df['Sex'] = pheno_df['Sex'].replace('PMI: Skip', 'Other')


In [ ]:
pheno_df["Sex"].value_counts()

In [ ]:
cols = ['PC1', 'PC16']

for col in cols:
    # 1. Force to string -> 2. Strip brackets -> 3. Convert to float
    pheno_df[col] = pheno_df[col].astype(str).str.replace(r'[\[\]]', '', regex=True).astype(float)

In [ ]:
pheno_df.to_csv(f"{output_dir}/results/polypharmacy_all_ancestry_multi_PGS.csv", index=False)